In [1]:
import numpy as np
import tensorflow as ts
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import load_model

In [2]:
## load word index of imdb dataset

word_index = imdb.get_word_index()
reverse_word_index = {value: key for (key, value) in word_index.items()}

reverse_word_index

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


{34701: 'fawn',
 52006: 'tsukino',
 52007: 'nunnery',
 16816: 'sonja',
 63951: 'vani',
 1408: 'woods',
 16115: 'spiders',
 2345: 'hanging',
 2289: 'woody',
 52008: 'trawling',
 52009: "hold's",
 11307: 'comically',
 40830: 'localized',
 30568: 'disobeying',
 52010: "'royale",
 40831: "harpo's",
 52011: 'canet',
 19313: 'aileen',
 52012: 'acurately',
 52013: "diplomat's",
 25242: 'rickman',
 6746: 'arranged',
 52014: 'rumbustious',
 52015: 'familiarness',
 52016: "spider'",
 68804: 'hahahah',
 52017: "wood'",
 40833: 'transvestism',
 34702: "hangin'",
 2338: 'bringing',
 40834: 'seamier',
 34703: 'wooded',
 52018: 'bravora',
 16817: 'grueling',
 1636: 'wooden',
 16818: 'wednesday',
 52019: "'prix",
 34704: 'altagracia',
 52020: 'circuitry',
 11585: 'crotch',
 57766: 'busybody',
 52021: "tart'n'tangy",
 14129: 'burgade',
 52023: 'thrace',
 11038: "tom's",
 52025: 'snuggles',
 29114: 'francesco',
 52027: 'complainers',
 52125: 'templarios',
 40835: '272',
 52028: '273',
 52130: 'zaniacs',

### Uploading the model file

Since the model file `simple_rnn_imdb.h5` is not found, we need to upload it to the Colab environment. Please run the following cell and select your `simple_rnn_imdb.h5` file from your local machine.

In [3]:
from google.colab import files

uploaded = files.upload()

Saving simple_rnn_imdb.h5 to simple_rnn_imdb.h5


After successfully uploading the file, you can re-run the cell below to load the model.

In [4]:
## load model

model = load_model('simple_rnn_imdb.h5')
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [ ]:
model.get_weights()

[array([[-0.6521489 , -0.44729754,  0.6879691 , ...,  0.514959  ,
          0.5718145 ,  0.5816156 ],
        [-0.02376965,  0.02419378,  0.02335258, ..., -0.01111971,
          0.07652581,  0.07220726],
        [-0.05089791, -0.01458984,  0.10878986, ...,  0.0740182 ,
          0.11858636,  0.01341708],
        ...,
        [ 0.00668954,  0.00795407,  0.00252768, ...,  0.03424724,
         -0.06538542, -0.04494979],
        [ 0.01756633, -0.06206906,  0.02653738, ...,  0.06294028,
          0.04799144,  0.0328557 ],
        [ 0.03928313,  0.00951008, -0.06966899, ..., -0.06034337,
         -0.09450869, -0.06032499]], dtype=float32),
 array([[-0.02068091,  0.18182592,  0.02004721, ...,  0.07805286,
          0.04766598,  0.03126881],
        [ 0.08982586,  0.04309196,  0.01878423, ..., -0.13996388,
          0.04333592,  0.06459653],
        [ 0.08720966, -0.02040574,  0.01469449, ..., -0.05507894,
         -0.11481483,  0.04149552],
        ...,
        [ 0.09378231,  0.11292129,  0.0

In [5]:
import string
from tensorflow.keras.preprocessing import sequence

## helper function

def decode_review(text):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in text])

def preprocess_text(text):
  # Clean the text: lowercase and remove punctuation
  text = text.lower()
  text = text.translate(str.maketrans('', '', string.punctuation))
  words = text.split()

  # Convert words to their integer representations, shifting by 3.
  # 0: padding, 1: start-of-sequence, 2: unknown
  # Words from word_index are 1-indexed. We add 3 to them.
  # Words not found in vocabulary map to 2 (unknown token).
  encoded_review = [word_index.get(word, 2) + 3 for word in words]

  # Add the start-of-sequence token (1) at the beginning
  encoded_review = [1] + encoded_review

  # Pad the sequence to maxlen, applying padding at the end
  return sequence.pad_sequences([encoded_review], maxlen=500, padding='post')

In [6]:
## prediction dunction

def predict_review(review):
  encoded_review = preprocess_text(review)
  prediction = model.predict(encoded_review)

  sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
  confidence = prediction[0][0] if sentiment == 'Positive' else 1 - prediction[0][0]

  return sentiment, confidence, prediction[0][0]

In [7]:
# predict text

example_text = "This movie was Fantastic! the acting was great and plot was thrilling."

sentiment, confidence, prediction = predict_review(example_text)

print(f"Sentiment: {sentiment}")
print(f"Confidence: {confidence:.2f}")
print(f"Prediction: {prediction}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 507ms/step
Sentiment: Positive
Confidence: 1.00
Prediction: 1.0


### Debugging Preprocessing

Let's look at how your `example_text` is encoded and then decoded to see what words the model is actually receiving. This can help identify if words are missing from the vocabulary or being incorrectly mapped.

In [8]:
original_text = example_text
encoded_sequence = preprocess_text(original_text)
print(f"Encoded Sequence: {encoded_sequence}")

# The preprocess_text returns a list of sequences, so we take the first one.
decoded_text = decode_review(encoded_sequence[0])

print(f"Original Text: {original_text}")
print(f"Decoded Text (as seen by model): {decoded_text}")

# Also check if important words are in the vocabulary
print("\nChecking important words in vocabulary:")
important_words = ['fantastic', 'great', 'thrilling']
for word in important_words:
    if word in word_index:
        print(f"'{word}' is in vocabulary with index {word_index[word]}")
    else:
        print(f"'{word}' is NOT in vocabulary")

Encoded Sequence: [[   1   14   20   16  777    4  116   16   87    5  114   16 3017    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0